# FoundationPose Evaluation — Pipeline Completo

**Un solo notebook. Ejecutar celda por celda.**

Pipeline:
1. Setup entorno + GPU
2. Datasets BOP (desde Drive cache)
3. Instalar FoundationPose + nvdiffrast
4. Descargar pesos pre-entrenados
5. Inferencia YCB-V (5 escenas, checkpoints en Drive)
6. Inferencia T-LESS (5 escenas, checkpoints en Drive)
7. Metricas + Comparativa vs GDR-Net++
8. Figuras para Capitulo 6

**Checkpoints:** Si Colab se desconecta, re-ejecuta todo. El setup toma ~5 min y la inferencia retoma automaticamente.

**Requiere:** GPU T4 (Runtime > Change runtime type > T4 GPU)

## 1. Setup Entorno

In [ ]:
import torch
import os
import sys
import time

# --- GPU ---
assert torch.cuda.is_available(), "GPU requerida! Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- Repo ---
REPO_URL = "https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git"
REPO_DIR = "/content/repo_tfm"

if not os.path.exists(REPO_DIR):
    print("Clonando repo...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Actualizando repo...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -1

# --- Dependencias ---
!pip install -q uv 2>&1 | tail -1
!uv pip install --system -q trimesh diffusers accelerate transformers scikit-learn

# Restaurar numpy si fue downgradeado
import numpy as np
if int(np.__version__.split('.')[0]) < 2:
    print(f"numpy {np.__version__} < 2.0, restaurando...")
    !uv pip install --system -q "numpy>=2.0"
    import importlib; importlib.reload(np)

print(f"numpy {np.__version__} OK")

# Verificar imports del proyecto
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc
from src.utils.dataset_loader import BOPDataset
print("Imports OK")

## 2. Google Drive + Datasets BOP

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_ZIPS = "/content/drive/MyDrive/TFM/datasets_zips"
LOCAL_DATA = "/content/datasets"
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights"

for d in [DRIVE_ZIPS, LOCAL_DATA, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Symlink: repo/data/datasets -> /content/datasets
REPO_DATA = f"{REPO_DIR}/data/datasets"
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA):
    shutil.rmtree(REPO_DATA)
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
os.symlink(LOCAL_DATA, REPO_DATA)
print(f"Symlink: {REPO_DATA} -> {LOCAL_DATA}")

# --- Funciones de descarga y extraccion ---
HF_BASE = "https://huggingface.co/datasets/bop-benchmark"

def download_and_extract(dataset, filename, url):
    zip_path = f"{DRIVE_ZIPS}/{filename}"
    extract_dir = f"{LOCAL_DATA}/{dataset}"
    os.makedirs(extract_dir, exist_ok=True)

    if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000:
        print(f"  [CACHE] {filename} ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    else:
        print(f"  [DOWN] {filename}...")
        !wget -q --show-progress -c -O "{zip_path}" "{url}"

    print(f"  [UNZIP] Extrayendo...")
    t0 = time.time()
    !unzip -qo "{zip_path}" -d "{extract_dir}"
    print(f"  [OK] {time.time()-t0:.0f}s")

def reorganize_bop(dataset_dir, nested_name, camera_src="camera.json"):
    """Aplana la estructura BOP: dataset/nested/ -> dataset/"""
    nested = Path(dataset_dir) / nested_name
    if nested.exists():
        !cp -rn {nested}/* {dataset_dir}/ 2>/dev/null; true
    cam_src = Path(dataset_dir) / camera_src
    if cam_src.exists() and not (Path(dataset_dir) / "camera.json").exists():
        !cp {cam_src} {dataset_dir}/camera.json

def verify_dataset(dataset_dir, split):
    """Verifica que el dataset tenga imagenes RGB."""
    test_dir = Path(dataset_dir) / split
    if not test_dir.exists():
        print(f"  ERROR: {split}/ no encontrado")
        return False
    scenes = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
    if not scenes:
        print(f"  ERROR: sin escenas en {split}/")
        return False
    rgb_dir = test_dir / scenes[0] / "rgb"
    imgs = sorted(rgb_dir.glob("*.*")) if rgb_dir.exists() else []
    print(f"  {len(scenes)} escenas, primera: {scenes[0]}, {len(imgs)} imgs ({imgs[0].name if imgs else 'N/A'})")
    return len(imgs) > 0

# --- YCB-Video ---
print("=== YCB-Video ===")
download_and_extract("ycbv", "ycbv_base.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_base.zip")
download_and_extract("ycbv", "ycbv_models.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_models.zip")
download_and_extract("ycbv", "ycbv_test_all.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_test_all.zip")

YCBV = f"{LOCAL_DATA}/ycbv"
reorganize_bop(YCBV, "ycbv", "camera_uw.json")
verify_dataset(YCBV, "test")

# --- T-LESS ---
print("\n=== T-LESS ===")
download_and_extract("tless", "tless_base.zip", f"{HF_BASE}/tless/resolve/main/tless_base.zip")
download_and_extract("tless", "tless_models.zip", f"{HF_BASE}/tless/resolve/main/tless_models.zip")
download_and_extract("tless", "tless_test_primesense_all.zip", f"{HF_BASE}/tless/resolve/main/tless_test_primesense_all.zip")

TLESS = f"{LOCAL_DATA}/tless"
reorganize_bop(TLESS, "tless", "camera_primesense.json")
# T-LESS usa models_cad como directorio de modelos
!ln -sf models_cad {TLESS}/models 2>/dev/null; true
verify_dataset(TLESS, "test_primesense")

# --- Espacio en disco ---
print("\n=== Disco ===")
!df -h /content | tail -1
print()
!du -sh {LOCAL_DATA}/*/ 2>/dev/null
print()
!du -sh {DRIVE_ZIPS}/ 2>/dev/null

## 3. Instalar FoundationPose

In [ ]:
FP_DIR = "/content/FoundationPose"

if not os.path.exists(FP_DIR):
    print("Clonando FoundationPose...")
    !git clone --depth 1 https://github.com/NVlabs/FoundationPose.git {FP_DIR}
else:
    print(f"FoundationPose ya existe en {FP_DIR}")

# ============================================================
# DEPENDENCIAS DEL SISTEMA (C++, OpenGL, CUDA)
# ============================================================
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ['PATH']}"

print("[1/6] Dependencias del sistema...")
!apt-get update -qq 2>&1 | tail -1
!apt-get install -q -y ninja-build \
    libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev libglvnd-dev \
    libboost-all-dev libeigen3-dev 2>&1 | tail -3

# ============================================================
# DEPENDENCIAS PYTHON
# ============================================================
print("\n[2/6] Dependencias Python...")
!pip install -q ninja cmake pybind11 \
    trimesh opencv-python-headless scipy scikit-learn \
    ruamel.yaml kornia transformations omegaconf einops

# ============================================================
# MOCK open3d (solo usado en debug, no en inferencia)
# ============================================================
print("\n[3/6] Mock open3d...")
import types
import numpy as _np

open3d_mock = types.ModuleType('open3d')
for sub in ['geometry', 'io', 'utility', 'visualization', 'core', 't', 'pipelines']:
    m = types.ModuleType(f'open3d.{sub}')
    setattr(open3d_mock, sub, m)
    sys.modules[f'open3d.{sub}'] = m

class _MockPC:
    def __init__(self):
        self.points = None
        self.colors = None
        self.normals = None
    def voxel_down_sample(self, voxel_size):
        if self.points is None: return self
        pts = _np.asarray(self.points)
        if len(pts) == 0: return self
        q = _np.round(pts / voxel_size).astype(_np.int32)
        _, idx = _np.unique(q, axis=0, return_index=True)
        r = _MockPC()
        r.points = open3d_mock.utility.Vector3dVector(pts[idx])
        if self.normals is not None:
            r.normals = open3d_mock.utility.Vector3dVector(_np.asarray(self.normals)[idx])
        return r
    def paint_uniform_color(self, c): return self
    def estimate_normals(self, **kw): pass
    def transform(self, T):
        if self.points is not None:
            pts = _np.asarray(self.points)
            pts_h = _np.c_[pts, _np.ones(len(pts))]
            pts_t = (T @ pts_h.T).T[:, :3]
            self.points = open3d_mock.utility.Vector3dVector(pts_t)
        return self

open3d_mock.geometry.PointCloud = _MockPC
open3d_mock.utility.Vector3dVector = lambda x: _np.asarray(x)
open3d_mock.io.write_point_cloud = lambda *a, **k: None
open3d_mock.io.read_point_cloud = lambda *a, **k: _MockPC()
sys.modules['open3d'] = open3d_mock
print("[OK] open3d mockeado (con voxel_down_sample)")

print("\n[4/6] nvdiffrast...")
try:
    import nvdiffrast
    print(f"[OK] nvdiffrast ya instalado")
except ImportError:
    !pip install nvdiffrast 2>&1 | tail -3
    try:
        import nvdiffrast
        print("[OK] nvdiffrast desde PyPI")
    except ImportError:
        print("PyPI fallo, compilando desde source...")
        !pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast 2>&1 | tail -10

# ============================================================
# PYTORCH3D
# ============================================================
print("\n[5/6] pytorch3d...")
try:
    import pytorch3d
    print(f"[OK] pytorch3d ya instalado")
except ImportError:
    print("Compilando pytorch3d desde source (~10-30 min)...")
    !pip install git+https://github.com/facebookresearch/pytorch3d.git@stable 2>&1 | tail -5

# ============================================================
# COMPILAR EXTENSIONES C++/CUDA DE FOUNDATIONPOSE
# ============================================================
print("\n[6/6] Compilando extensiones FoundationPose...")

# mycpp (C++ con pybind11 + boost + eigen)
mycpp_dir = f"{FP_DIR}/mycpp"
mycpp_build = f"{mycpp_dir}/build"
if os.path.exists(mycpp_dir) and not os.path.exists(f"{mycpp_build}/mycpp.cpython*.so"):
    print("  Compilando mycpp...")
    os.makedirs(mycpp_build, exist_ok=True)
    !cd {mycpp_build} && cmake .. 2>&1 | tail -3
    !cd {mycpp_build} && make -j$(nproc) 2>&1 | tail -3
    print("  [OK] mycpp")
else:
    print("  mycpp: ya compilado o no encontrado")

# ---- Fallback Python para mycpp.cluster_poses ----
# mycpp C++ falla en Colab, pero cluster_poses es la unica funcion usada.
# Implementamos un fallback puro en Python/NumPy.
try:
    import mycpp.build.mycpp as mycpp
    print("  [OK] mycpp importado")
except Exception:
    print("  [WARN] mycpp C++ no disponible, usando fallback Python")
    import types as _types
    _mycpp_mod = _types.ModuleType('mycpp')
    _mycpp_build = _types.ModuleType('mycpp.build')
    _mycpp_inner = _types.ModuleType('mycpp.build.mycpp')

    def _rotation_geodesic(R1, R2):
        cos_a = ((_np.trace(R1 @ R2.T) - 1.0) / 2.0)
        cos_a = _np.clip(cos_a, -1.0, 1.0)
        return _np.arccos(cos_a)

    def _cluster_poses(angle_diff_deg, dist_diff, poses_in, symmetry_tfs):
        """Greedy pose clustering respecting object symmetries."""
        radian_thres = angle_diff_deg / 180.0 * _np.pi
        poses_in = _np.asarray(poses_in)
        symmetry_tfs = _np.asarray(symmetry_tfs)
        if len(poses_in) == 0:
            return poses_in
        out = [poses_in[0]]
        for i in range(1, len(poses_in)):
            cur = poses_in[i]
            is_new = True
            t1 = cur[:3, 3]
            for c in out:
                t0 = c[:3, 3]
                if _np.linalg.norm(t0 - t1) >= dist_diff:
                    continue
                for tf in symmetry_tfs:
                    cur_sym = cur @ tf
                    rd = _rotation_geodesic(cur_sym[:3, :3], c[:3, :3])
                    if rd < radian_thres:
                        is_new = False
                        break
                if not is_new:
                    break
            if is_new:
                out.append(cur)
        return _np.array(out)

    _mycpp_inner.cluster_poses = _cluster_poses
    _mycpp_build.mycpp = _mycpp_inner
    _mycpp_mod.build = _mycpp_build
    sys.modules['mycpp'] = _mycpp_mod
    sys.modules['mycpp.build'] = _mycpp_build
    sys.modules['mycpp.build.mycpp'] = _mycpp_inner
    print("  [OK] mycpp fallback Python registrado")


# mycuda (CUDA extensions)
import glob
mycuda_dirs = glob.glob(f"{FP_DIR}/**/mycuda", recursive=True)
for d in mycuda_dirs:
    setup_file = os.path.join(d, 'setup.py')
    if os.path.exists(setup_file):
        print(f"  Compilando {d}...")
        !cd {d} && python setup.py build_ext --inplace 2>&1 | tail -3

sys.path.insert(0, FP_DIR)

# ============================================================
# VERIFICACION FINAL
# ============================================================
print("\n" + "=" * 50)
print("VERIFICACION DE DEPENDENCIAS")
print("=" * 50)

import numpy as np
if int(np.__version__.split('.')[0]) < 2:
    !pip install -q "numpy>=2.0"
    import importlib; importlib.reload(np)
print(f"numpy: {np.__version__}")

checks = [
    ('nvdiffrast', 'import nvdiffrast.torch'),
    ('pytorch3d', 'import pytorch3d'),
    ('kornia', 'import kornia'),
    ('transformations', 'import transformations'),
    ('trimesh', 'import trimesh'),
    ('open3d (mock)', 'import open3d'),
]
for name, cmd in checks:
    try:
        exec(cmd)
        print(f"[OK] {name}")
    except Exception as e:
        print(f"[ERROR] {name}: {e}")

# Test FP import
print("\n--- Test import FoundationPose ---")
try:
    from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
    print("[OK] FoundationPose importado correctamente")
except Exception as e:
    print(f"[ERROR] FoundationPose: {e}")
    print("Revisa los errores arriba e instala lo que falte.")

print(f"\nFoundationPose en: {FP_DIR}")

## 4. Pesos Pre-entrenados

In [ ]:
!pip install -q gdown
import gdown

# --- Descargar pesos desde Google Drive oficial de NVIDIA ---
# Ref: https://github.com/NVlabs/FoundationPose#pre-trained-weights
# Folder: https://drive.google.com/drive/folders/1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i

SCORER_DIR = f"{FP_DIR}/weights/2024-01-11-20-02-45"
REFINER_DIR = f"{FP_DIR}/bundlesdf/ckpts/2023-10-28-18-33-37"
GDRIVE_WEIGHTS = "/content/drive/MyDrive/TFM/weights/foundationpose"
os.makedirs(GDRIVE_WEIGHTS, exist_ok=True)

def download_fp_weights():
    """Descargar pesos de FoundationPose desde Google Drive oficial."""
    # Google Drive folder ID de NVIDIA
    WEIGHTS_FOLDER_ID = "1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i"
    weights_zip = f"{GDRIVE_WEIGHTS}/fp_weights.zip"
    
    # Verificar si ya tenemos los pesos en Drive (cache)
    if os.path.exists(SCORER_DIR) and os.path.exists(REFINER_DIR):
        scorer_files = os.listdir(SCORER_DIR)
        refiner_files = os.listdir(REFINER_DIR)
        if 'config.yml' in scorer_files and len(refiner_files) > 0:
            print(f"[OK] Scorer: {len(scorer_files)} archivos")
            print(f"[OK] Refiner: {len(refiner_files)} archivos")
            return True
    
    # Verificar si los pesos estan en Drive (de sesion anterior)
    drive_scorer = f"{GDRIVE_WEIGHTS}/2024-01-11-20-02-45"
    drive_refiner = f"{GDRIVE_WEIGHTS}/2023-10-28-18-33-37"
    if os.path.exists(drive_scorer) and os.path.exists(drive_refiner):
        print("Copiando pesos desde Drive (cache)...")
        import shutil
        os.makedirs(os.path.dirname(SCORER_DIR), exist_ok=True)
        os.makedirs(os.path.dirname(REFINER_DIR), exist_ok=True)
        if not os.path.exists(SCORER_DIR):
            shutil.copytree(drive_scorer, SCORER_DIR)
        if not os.path.exists(REFINER_DIR):
            shutil.copytree(drive_refiner, REFINER_DIR)
        print("[OK] Pesos copiados desde Drive")
        return True
    
    # Descargar desde Google Drive de NVIDIA
    print("Descargando pesos desde Google Drive de NVIDIA...")
    print("(Primera vez, ~500 MB, se cachea en Drive para futuras sesiones)")
    
    # Descargar toda la carpeta de pesos
    download_dir = f"{GDRIVE_WEIGHTS}/download_temp"
    os.makedirs(download_dir, exist_ok=True)
    
    try:
        gdown.download_folder(
            url=f"https://drive.google.com/drive/folders/{WEIGHTS_FOLDER_ID}",
            output=download_dir,
            quiet=False
        )
    except Exception as e:
        print(f"Error con gdown: {e}")
        print("\nDescarga manual necesaria:")
        print(f"1. Ve a: https://drive.google.com/drive/folders/{WEIGHTS_FOLDER_ID}")
        print(f"2. Descarga las carpetas 2024-01-11-20-02-45 y 2023-10-28-18-33-37")
        print(f"3. Subelas a: {GDRIVE_WEIGHTS}/")
        return False
    
    # Mover a las ubicaciones correctas
    import shutil
    downloaded = os.listdir(download_dir)
    print(f"Descargado: {downloaded}")
    
    for item in downloaded:
        src = os.path.join(download_dir, item)
        if '2024-01-11' in item:  # Scorer
            dst_drive = drive_scorer
            dst_fp = SCORER_DIR
        elif '2023-10-28' in item:  # Refiner
            dst_drive = drive_refiner
            dst_fp = REFINER_DIR
        else:
            continue
        
        # Guardar en Drive (cache persistente)
        if os.path.isdir(src) and not os.path.exists(dst_drive):
            shutil.copytree(src, dst_drive)
            print(f"  Guardado en Drive: {dst_drive}")
        
        # Copiar al repo de FP
        os.makedirs(os.path.dirname(dst_fp), exist_ok=True)
        if not os.path.exists(dst_fp):
            shutil.copytree(src, dst_fp)
            print(f"  Copiado a FP: {dst_fp}")
    
    # Limpiar temp
    shutil.rmtree(download_dir, ignore_errors=True)
    return True

success = download_fp_weights()

# PoseRefinePredictor busca en weights/, no en bundlesdf/ckpts/
refiner_in_weights = f"{FP_DIR}/weights/2023-10-28-18-33-37"
if os.path.exists(REFINER_DIR) and not os.path.exists(refiner_in_weights):
    import shutil
    shutil.copytree(REFINER_DIR, refiner_in_weights)
    print(f"Refiner copiado a {refiner_in_weights}")

if success:
    print("\n" + "=" * 50)
    print("[OK] Pesos de FoundationPose listos")
    print("=" * 50)
    !ls {SCORER_DIR}/ 2>/dev/null | head -5
    !ls {REFINER_DIR}/ 2>/dev/null | head -5
else:
    print("\n[!] Pesos no disponibles. Sigue las instrucciones arriba.")

## 5. Inicializar FoundationPose

In [ ]:
import trimesh
import nvdiffrast.torch as dr

# Importar modulos de FoundationPose
sys.path.insert(0, FP_DIR)
from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor

# --- Inicializar los modelos (una sola vez) ---
print("Cargando ScorePredictor...")
scorer = ScorePredictor()
print("[OK] ScorePredictor cargado")

print("Cargando PoseRefinePredictor...")
refiner = PoseRefinePredictor()
print("[OK] PoseRefinePredictor cargado")

print("Creando contexto de rasterizacion CUDA...")
glctx = dr.RasterizeCudaContext()
print("[OK] nvdiffrast glctx creado")

print("\nFoundationPose listo para inferencia.")

In [ ]:
# --- Cargar meshes de los objetos ---
# BOP format: data/datasets/{ycbv,tless}/models/obj_XXXXXX.ply

import json

DATA_DIR = f"{REPO_DIR}/data/datasets"

def load_bop_meshes(dataset_dir):
    """Carga todos los meshes de un dataset BOP.

    Returns:
        dict: {obj_id: trimesh.Trimesh}
    """
    models_dir = Path(dataset_dir) / 'models'
    meshes = {}

    for ply_file in sorted(models_dir.glob('obj_*.ply')):
        # obj_000001.ply -> obj_id = 1
        obj_id = int(ply_file.stem.split('_')[1])
        mesh = trimesh.load(str(ply_file), process=False)

        # BOP meshes estan en mm, convertir a metros
        mesh.vertices *= 1e-3

        meshes[obj_id] = mesh

    return meshes


def create_estimator(mesh, scorer, refiner, glctx):
    """Crea un estimador FoundationPose para un objeto dado."""
    est = FoundationPose(
        model_pts=mesh.vertices.copy(),
        model_normals=mesh.vertex_normals.copy(),
        mesh=mesh,
        scorer=scorer,
        refiner=refiner,
        glctx=glctx,
        debug=0,
    )
    return est


# Cargar meshes YCB-V
ycbv_meshes = load_bop_meshes(f"{DATA_DIR}/ycbv")
print(f"YCB-V: {len(ycbv_meshes)} meshes cargados (obj_ids: {sorted(ycbv_meshes.keys())})")

# Cargar meshes T-LESS
tless_path = Path(DATA_DIR) / 'tless'
if (tless_path / 'models').exists():
    tless_meshes = load_bop_meshes(f"{DATA_DIR}/tless")
    print(f"T-LESS: {len(tless_meshes)} meshes cargados")
else:
    tless_meshes = {}
    print("T-LESS: modelos no disponibles")

# --- Configuracion de evaluacion ---
MAX_SCENES = 5
MAX_IMAGES_PER_SCENE = 50
REGISTER_ITERATIONS = 5

## 6. Inferencia YCB-Video

In [ ]:
from src.utils.dataset_loader import BOPDataset
from tqdm.notebook import tqdm
import cv2

# Cargar dataset YCB-V
print(f"Cargando YCB-V desde {DATA_DIR}/ycbv")
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
ycbv_scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(ycbv_scenes)} escenas de test")

eval_scenes = ycbv_scenes[:MAX_SCENES] if MAX_SCENES else ycbv_scenes
print(f"\nEvaluando {len(eval_scenes)} escenas (MAX_SCENES={MAX_SCENES})")
print(f"Iteraciones de registro: {REGISTER_ITERATIONS}")

# --- Inferencia con CHECKPOINTS ---
CHECKPOINT_EVERY = 2
CHECKPOINT_DIR = "/content/drive/MyDrive/TFM/checkpoints"
CHECKPOINT_FILE = f"{CHECKPOINT_DIR}/fp_ycbv_checkpoint.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Intentar retomar desde checkpoint
results_ycbv = []
completed_scenes = set()
timing_total = 0
n_objects_evaluated = 0
n_images_evaluated = 0
errors = []

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    if ckpt.get("n_objects_evaluated", 0) > 0:
        results_ycbv = ckpt["results"]
        completed_scenes = set(ckpt["completed_scenes"])
        timing_total = ckpt["timing_total"]
        n_objects_evaluated = ckpt["n_objects_evaluated"]
        n_images_evaluated = ckpt["n_images_evaluated"]
        print(f"CHECKPOINT VALIDO: {len(completed_scenes)} escenas, {n_objects_evaluated} objetos")
        print(f"  Retomando desde escena siguiente...\n")
    else:
        print("Checkpoint anterior con 0 predicciones (descartado). Reiniciando.\n")
        os.remove(CHECKPOINT_FILE)
else:
    print("Sin checkpoint previo. Comenzando desde cero.\n")

estimator_cache = {}

for scene_idx, scene_id in enumerate(tqdm(eval_scenes, desc="Escenas YCB-V")):
    # Saltar escenas ya completadas
    if scene_id in completed_scenes:
        continue

    scene_gt = ycbv.load_scene_gt(scene_id)
    scene_camera = ycbv.load_scene_camera(scene_id)

    image_ids = ycbv.get_image_ids(scene_id)
    for img_id in image_ids[:MAX_IMAGES_PER_SCENE]:
        try:
            sample = ycbv.load_sample(scene_id, img_id)
            rgb = sample["rgb"]
            depth = sample["depth"]
            cam_K = sample["cam_K"]

            img_key = str(img_id)
            gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

            for gt_idx, gt in enumerate(gt_list):
                obj_id = gt["obj_id"]
                if obj_id not in ycbv_meshes:
                    continue

                mask = sample.get("mask", None)
                if mask is None:
                    mask_path = Path(DATA_DIR) / "ycbv" / "test" / scene_id / "mask_visib" / f"{img_id:06d}_{gt_idx:06d}.png"
                    if mask_path.exists():
                        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                        mask = (mask > 0).astype(np.uint8)
                    else:
                        continue
                elif mask.ndim == 2:
                    mask = (mask > 0).astype(np.uint8)

                if obj_id not in estimator_cache:
                    estimator_cache[obj_id] = create_estimator(
                        ycbv_meshes[obj_id], scorer, refiner, glctx
                    )
                est = estimator_cache[obj_id]

                t0 = time.time()
                pose_4x4 = est.register(
                    K=cam_K, rgb=rgb, depth=depth,
                    ob_mask=mask, iteration=REGISTER_ITERATIONS,
                )
                elapsed = time.time() - t0
                timing_total += elapsed
                n_objects_evaluated += 1

                R_pred = pose_4x4[:3, :3]
                t_pred = pose_4x4[:3, 3]

                results_ycbv.append({
                    "scene_id": scene_id,
                    "img_id": int(img_id),
                    "obj_id": int(obj_id),
                    "R_pred": R_pred.tolist(),
                    "t_pred": t_pred.tolist(),
                    "time_s": elapsed,
                })

            n_images_evaluated += 1

        except Exception as e:
            err_msg = f"Scene {scene_id} img {img_id}: {str(e)[:150]}"
            errors.append(err_msg)
            if len(errors) <= 5:
                print(f"\n[Error] {err_msg}")
            continue

    completed_scenes.add(scene_id)

    # --- CHECKPOINT: guardar progreso ---
    if (scene_idx + 1) % CHECKPOINT_EVERY == 0:
        ckpt_data = {
            "results": results_ycbv,
            "completed_scenes": list(completed_scenes),
            "timing_total": timing_total,
            "n_objects_evaluated": n_objects_evaluated,
            "n_images_evaluated": n_images_evaluated,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(ckpt_data, f)
        print(f"  [CKPT] Guardado: {len(completed_scenes)} escenas, {n_objects_evaluated} objetos")

# Checkpoint final
if results_ycbv:
    ckpt_data = {
        "results": results_ycbv,
        "completed_scenes": list(completed_scenes),
        "timing_total": timing_total,
        "n_objects_evaluated": n_objects_evaluated,
        "n_images_evaluated": n_images_evaluated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "COMPLETED",
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(ckpt_data, f)

# --- Resumen ---
avg_time = timing_total / max(n_objects_evaluated, 1)
print(f"\n{'=' * 60}")
print(f"  FoundationPose YCB-Video -- Resumen")
print(f"{'=' * 60}")
print(f"  Imagenes evaluadas:    {n_images_evaluated}")
print(f"  Objetos evaluados:     {n_objects_evaluated}")
print(f"  Predicciones totales:  {len(results_ycbv)}")
print(f"  Tiempo promedio/obj:   {avg_time*1000:.1f} ms")
if avg_time > 0:
    print(f"  Throughput:            {1/avg_time:.1f} objetos/s")
if errors:
    print(f"  Errores:               {len(errors)}")
print(f"  Checkpoint:            {CHECKPOINT_FILE}")
print(f"{'=' * 60}")


## 7. Metricas YCB-Video

In [ ]:
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc

if results_ycbv:
    # Calcular ADD y ADD-S comparando con GT
    add_errors = []
    adds_errors = []

    for pred in tqdm(results_ycbv, desc="Calculando metricas YCB-V"):
        scene_id = pred['scene_id']
        img_id = pred['img_id']
        obj_id = pred['obj_id']

        # Obtener GT pose
        scene_gt = ycbv.load_scene_gt(scene_id)
        img_key = str(img_id)
        gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

        gt_pose = None
        for gt in gt_list:
            if gt['obj_id'] == obj_id:
                R_gt = np.array(gt['cam_R_m2c']).reshape(3, 3)
                t_gt = np.array(gt['cam_t_m2c']).reshape(3) * 1e-3  # mm -> m
                gt_pose = (R_gt, t_gt)
                break

        if gt_pose is None:
            continue

        R_pred = np.array(pred['R_pred'])
        t_pred = np.array(pred['t_pred'])

        # Puntos del modelo
        if obj_id in ycbv_meshes:
            pts = ycbv_meshes[obj_id].vertices  # Ya en metros

            # ADD
            add_err = add_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            add_errors.append(add_err)

            # ADD-S (simetrico)
            adds_err = add_s_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            adds_errors.append(adds_err)

    # Calcular recall y AUC
    add_errors = np.array(add_errors)
    adds_errors = np.array(adds_errors)

    thresholds_mm = [5, 10, 20, 50]  # en mm

    print(f"\n{'=' * 60}")
    print(f"  FoundationPose -- YCB-Video Metrics")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados: {len(add_errors)}")
    print(f"")
    print(f"  ADD  (mean):     {add_errors.mean()*1000:.2f} mm")
    print(f"  ADD  (median):   {np.median(add_errors)*1000:.2f} mm")
    print(f"  ADD-S (mean):    {adds_errors.mean()*1000:.2f} mm")
    print(f"  ADD-S (median):  {np.median(adds_errors)*1000:.2f} mm")
    print(f"")

    for th in thresholds_mm:
        recall_add = (add_errors * 1000 < th).mean() * 100
        recall_adds = (adds_errors * 1000 < th).mean() * 100
        print(f"  Recall@{th}mm  ADD: {recall_add:.1f}%  ADD-S: {recall_adds:.1f}%")

    # AUC (area bajo la curva recall vs threshold)
    auc_add = compute_auc(list(add_errors * 1000), max_threshold=100)
    auc_adds = compute_auc(list(adds_errors * 1000), max_threshold=100)
    print(f"")
    print(f"  AUC ADD:   {auc_add:.4f}")
    print(f"  AUC ADD-S: {auc_adds:.4f}")
    print(f"{'=' * 60}")

    # Guardar metricas en dict
    metrics_fp_ycbv = {
        'add_mean_mm': float(add_errors.mean() * 1000),
        'add_median_mm': float(np.median(add_errors) * 1000),
        'adds_mean_mm': float(adds_errors.mean() * 1000),
        'adds_median_mm': float(np.median(adds_errors) * 1000),
        'auc_add': float(auc_add),
        'auc_adds': float(auc_adds),
        'n_evaluated': len(add_errors),
    }
    for th in thresholds_mm:
        metrics_fp_ycbv[f'recall_add_{th}mm'] = float((add_errors * 1000 < th).mean())
        metrics_fp_ycbv[f'recall_adds_{th}mm'] = float((adds_errors * 1000 < th).mean())
else:
    print("Sin predicciones YCB-V para calcular metricas.")
    metrics_fp_ycbv = {}

## 8. Inferencia T-LESS

In [ ]:
TLESS_CKPT_FILE = f"{CHECKPOINT_DIR}/fp_tless_checkpoint.json"

# Retomar desde checkpoint si existe
results_tless = []
completed_tless_scenes = set()
timing_tless = 0
n_tless_obj = 0
n_tless_images = 0
tless_errors = []

if os.path.exists(TLESS_CKPT_FILE):
    with open(TLESS_CKPT_FILE) as f:
        ckpt = json.load(f)
    if ckpt.get("n_objects_evaluated", 0) > 0:
        results_tless = ckpt["results"]
        completed_tless_scenes = set(ckpt["completed_scenes"])
        timing_tless = ckpt["timing_total"]
        n_tless_obj = ckpt["n_objects_evaluated"]
        n_tless_images = ckpt.get("n_images_evaluated", 0)
        print(f"CHECKPOINT T-LESS VALIDO: {len(completed_tless_scenes)} escenas, {n_tless_obj} objetos")
    else:
        print("Checkpoint T-LESS anterior con 0 predicciones (descartado). Reiniciando.\n")
        os.remove(TLESS_CKPT_FILE)
else:
    print("Sin checkpoint T-LESS previo. Comenzando desde cero.\n")

tless_test_path = Path(f"{DATA_DIR}/tless/test_primesense")

if tless_test_path.exists() and tless_meshes:
    tless = BOPDataset(f"{DATA_DIR}/tless", split="test_primesense")
    tless_scenes = tless.get_scene_ids()
    print(f"T-LESS: {len(tless_scenes)} escenas de test")

    eval_tless = tless_scenes[:MAX_SCENES]
    print(f"Evaluando {len(eval_tless)} escenas (MAX_SCENES={MAX_SCENES})\n")

    tless_estimator_cache = {}

    for scene_idx, scene_id in enumerate(tqdm(eval_tless, desc="Escenas T-LESS")):
        if scene_id in completed_tless_scenes:
            continue

        scene_gt = tless.load_scene_gt(scene_id)

        image_ids = tless.get_image_ids(scene_id)
        for img_id in image_ids[:MAX_IMAGES_PER_SCENE]:
            try:
                sample = tless.load_sample(scene_id, img_id)
                rgb = sample["rgb"]
                depth = sample["depth"]
                cam_K = sample["cam_K"]

                img_key = str(img_id)
                gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

                for gt_idx, gt in enumerate(gt_list):
                    obj_id = gt["obj_id"]
                    if obj_id not in tless_meshes:
                        continue

                    mask_path = Path(DATA_DIR) / "tless" / "test_primesense" / scene_id / "mask_visib" / f"{img_id:06d}_{gt_idx:06d}.png"
                    if mask_path.exists():
                        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                        mask = (mask > 0).astype(np.uint8)
                    else:
                        continue

                    if obj_id not in tless_estimator_cache:
                        tless_estimator_cache[obj_id] = create_estimator(
                            tless_meshes[obj_id], scorer, refiner, glctx
                        )
                    est = tless_estimator_cache[obj_id]

                    t0 = time.time()
                    pose_4x4 = est.register(
                        K=cam_K, rgb=rgb, depth=depth,
                        ob_mask=mask, iteration=REGISTER_ITERATIONS,
                    )
                    elapsed = time.time() - t0
                    timing_tless += elapsed
                    n_tless_obj += 1

                    results_tless.append({
                        "scene_id": scene_id,
                        "img_id": int(img_id),
                        "obj_id": int(obj_id),
                        "R_pred": pose_4x4[:3, :3].tolist(),
                        "t_pred": pose_4x4[:3, 3].tolist(),
                        "time_s": elapsed,
                    })

                n_tless_images += 1

            except Exception as e:
                err_msg = f"Scene {scene_id} img {img_id}: {str(e)[:150]}"
                tless_errors.append(err_msg)
                if len(tless_errors) <= 5:
                    print(f"\n[Error] {err_msg}")
                continue

        completed_tless_scenes.add(scene_id)

        # --- CHECKPOINT ---
        if (scene_idx + 1) % CHECKPOINT_EVERY == 0:
            ckpt_data = {
                "results": results_tless,
                "completed_scenes": list(completed_tless_scenes),
                "timing_total": timing_tless,
                "n_objects_evaluated": n_tless_obj,
                "n_images_evaluated": n_tless_images,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            }
            with open(TLESS_CKPT_FILE, "w") as f:
                json.dump(ckpt_data, f)
            print(f"  [CKPT] {len(completed_tless_scenes)} escenas, {n_tless_obj} objetos")

    # Checkpoint final
    if results_tless:
        ckpt_data = {
            "results": results_tless,
            "completed_scenes": list(completed_tless_scenes),
            "timing_total": timing_tless,
            "n_objects_evaluated": n_tless_obj,
            "n_images_evaluated": n_tless_images,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "status": "COMPLETED",
        }
        with open(TLESS_CKPT_FILE, "w") as f:
            json.dump(ckpt_data, f)

    avg_tless = timing_tless / max(n_tless_obj, 1)
    print(f"\n{'=' * 60}")
    print(f"  FoundationPose T-LESS -- Resumen")
    print(f"{'=' * 60}")
    print(f"  Imagenes evaluadas:    {n_tless_images}")
    print(f"  Objetos evaluados:     {n_tless_obj}")
    print(f"  Predicciones totales:  {len(results_tless)}")
    print(f"  Tiempo promedio/obj:   {avg_tless*1000:.1f} ms")
    if avg_tless > 0:
        print(f"  Throughput:            {1/avg_tless:.1f} objetos/s")
    if tless_errors:
        print(f"  Errores:               {len(tless_errors)}")
    print(f"  Checkpoint:            {TLESS_CKPT_FILE}")
    print(f"{'=' * 60}")
else:
    print("T-LESS no disponible. Verifica la descarga del dataset.")


## 9. Metricas T-LESS

In [ ]:
if results_tless:
    # Calcular ADD y ADD-S comparando con GT
    add_errors_tless = []
    adds_errors_tless = []

    for pred in tqdm(results_tless, desc="Calculando metricas T-LESS"):
        scene_id = pred['scene_id']
        img_id = pred['img_id']
        obj_id = pred['obj_id']

        # Obtener GT pose
        scene_gt = tless.load_scene_gt(scene_id)
        img_key = str(img_id)
        gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

        gt_pose = None
        for gt in gt_list:
            if gt['obj_id'] == obj_id:
                R_gt = np.array(gt['cam_R_m2c']).reshape(3, 3)
                t_gt = np.array(gt['cam_t_m2c']).reshape(3) * 1e-3  # mm -> m
                gt_pose = (R_gt, t_gt)
                break

        if gt_pose is None:
            continue

        R_pred = np.array(pred['R_pred'])
        t_pred = np.array(pred['t_pred'])

        # Puntos del modelo (ADD-S es mas apropiado para T-LESS por simetrias)
        if obj_id in tless_meshes:
            pts = tless_meshes[obj_id].vertices  # Ya en metros

            # ADD
            add_err = add_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            add_errors_tless.append(add_err)

            # ADD-S (simetrico)
            adds_err = add_s_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            adds_errors_tless.append(adds_err)

    add_errors_tless = np.array(add_errors_tless)
    adds_errors_tless = np.array(adds_errors_tless)

    thresholds_mm = [5, 10, 20, 50]

    print(f"\n{'=' * 60}")
    print(f"  FoundationPose -- T-LESS Metrics")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados: {len(add_errors_tless)}")
    print(f"")
    print(f"  ADD  (mean):     {add_errors_tless.mean()*1000:.2f} mm")
    print(f"  ADD  (median):   {np.median(add_errors_tless)*1000:.2f} mm")
    print(f"  ADD-S (mean):    {adds_errors_tless.mean()*1000:.2f} mm")
    print(f"  ADD-S (median):  {np.median(adds_errors_tless)*1000:.2f} mm")
    print(f"")

    for th in thresholds_mm:
        recall_add = (add_errors_tless * 1000 < th).mean() * 100
        recall_adds = (adds_errors_tless * 1000 < th).mean() * 100
        print(f"  Recall@{th}mm  ADD: {recall_add:.1f}%  ADD-S: {recall_adds:.1f}%")

    auc_add_tless = compute_auc(list(add_errors_tless * 1000), max_threshold=100)
    auc_adds_tless = compute_auc(list(adds_errors_tless * 1000), max_threshold=100)
    print(f"")
    print(f"  AUC ADD:   {auc_add_tless:.4f}")
    print(f"  AUC ADD-S: {auc_adds_tless:.4f}")
    print(f"{'=' * 60}")

    metrics_fp_tless = {
        'add_mean_mm': float(add_errors_tless.mean() * 1000),
        'add_median_mm': float(np.median(add_errors_tless) * 1000),
        'adds_mean_mm': float(adds_errors_tless.mean() * 1000),
        'adds_median_mm': float(np.median(adds_errors_tless) * 1000),
        'auc_add': float(auc_add_tless),
        'auc_adds': float(auc_adds_tless),
        'n_evaluated': len(add_errors_tless),
    }
    for th in thresholds_mm:
        metrics_fp_tless[f'recall_add_{th}mm'] = float((add_errors_tless * 1000 < th).mean())
        metrics_fp_tless[f'recall_adds_{th}mm'] = float((adds_errors_tless * 1000 < th).mean())
else:
    print("Sin predicciones T-LESS para calcular metricas.")
    metrics_fp_tless = {}

## 10. Comparativa vs GDR-Net++

In [ ]:
# --- Baselines de referencia ---
# GDR-Net++ (BOP Challenge 2022 winner)
# Fuente: Hodan et al., BOP Challenge 2022, Table 3
#   https://bop.felk.cvut.cz/leaderboards/bop-challenge-2022/

gdrnet_baselines = {
    'ycbv': {
        'method': 'GDR-Net++',
        'source': 'BOP Challenge 2022',
        'vsd_recall': 0.842,
        'mssd_recall': 0.819,
        'mspd_recall': 0.874,
        'bop_avg_recall': 0.845,
    },
    'tless': {
        'method': 'GDR-Net++',
        'source': 'BOP Challenge 2022',
        'vsd_recall': 0.736,
        'mssd_recall': 0.685,
        'mspd_recall': 0.773,
        'bop_avg_recall': 0.731,
    }
}

# FoundationPose paper results (Wen et al., CVPR 2024, Table 1)
fp_paper_baselines = {
    'ycbv': {
        'method': 'FoundationPose (paper)',
        'source': 'Wen et al., CVPR 2024',
        'vsd_recall': 0.882,
        'mssd_recall': 0.862,
        'mspd_recall': 0.907,
        'bop_avg_recall': 0.884,
    },
    'tless': {
        'method': 'FoundationPose (paper)',
        'source': 'Wen et al., CVPR 2024',
        'vsd_recall': 0.774,
        'mssd_recall': 0.725,
        'mspd_recall': 0.832,
        'bop_avg_recall': 0.777,
    }
}

# --- Tabla comparativa ---
print(f"{'=' * 80}")
print(f"  COMPARATIVA: FoundationPose vs GDR-Net++")
print(f"{'=' * 80}")
print()

header = f"{'Metodo':<30} {'VSD':>8} {'MSSD':>8} {'MSPD':>8} {'BOP Avg':>8}"
sep = '-' * len(header)

for ds_name in ['ycbv', 'tless']:
    print(f"\n  Dataset: {ds_name.upper()}")
    print(f"  {sep}")
    print(f"  {header}")
    print(f"  {sep}")

    # GDR-Net++
    g = gdrnet_baselines[ds_name]
    print(f"  {g['method']:<30} {g['vsd_recall']:>8.3f} {g['mssd_recall']:>8.3f} {g['mspd_recall']:>8.3f} {g['bop_avg_recall']:>8.3f}")

    # FP paper
    fp = fp_paper_baselines[ds_name]
    print(f"  {fp['method']:<30} {fp['vsd_recall']:>8.3f} {fp['mssd_recall']:>8.3f} {fp['mspd_recall']:>8.3f} {fp['bop_avg_recall']:>8.3f}")

    # Nuestros resultados (ADD/ADD-S como proxy)
    our_metrics = metrics_fp_ycbv if ds_name == 'ycbv' else metrics_fp_tless
    if our_metrics:
        print(f"  {'FP (ours, ADD-based)':<30} {'---':>8} {'---':>8} {'---':>8} {'---':>8}")
        print(f"    ADD AUC: {our_metrics.get('auc_add', 0):.4f}")
        print(f"    ADD-S AUC: {our_metrics.get('auc_adds', 0):.4f}")
        print(f"    N evaluated: {our_metrics.get('n_evaluated', 0)}")
    else:
        print(f"  {'FP (ours)':<30} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>8}")

    print(f"  {sep}")

print(f"\n{'=' * 80}")
print("  Nota: VSD/MSSD/MSPD de GDR-Net++ y FP (paper) son del BOP leaderboard.")
print("  Nuestros resultados usan ADD/ADD-S en un subconjunto (5 escenas x 50 imgs).")
print("  Para comparacion directa, enviar al BOP evaluation server.")
print(f"{'=' * 80}")

## 11. Figuras Capitulo 6

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

FIGURES_DIR = "/content/drive/MyDrive/TFM/figures/chapter6"
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Bar chart: FP paper vs GDR-Net++ on BOP metrics ---
datasets = ['YCB-V', 'T-LESS']
metrics_names = ['VSD', 'MSSD', 'MSPD']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ds_label, ds_key) in enumerate(zip(datasets, ['ycbv', 'tless'])):
    ax = axes[ax_idx]

    gdr_vals = [
        gdrnet_baselines[ds_key]['vsd_recall'],
        gdrnet_baselines[ds_key]['mssd_recall'],
        gdrnet_baselines[ds_key]['mspd_recall'],
    ]
    fp_vals = [
        fp_paper_baselines[ds_key]['vsd_recall'],
        fp_paper_baselines[ds_key]['mssd_recall'],
        fp_paper_baselines[ds_key]['mspd_recall'],
    ]

    x = np.arange(len(metrics_names))
    width = 0.35

    bars1 = ax.bar(x - width/2, [v * 100 for v in gdr_vals], width,
                   label='GDR-Net++ (BOP 2022)', color='#4C72B0', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, [v * 100 for v in fp_vals], width,
                   label='FoundationPose (CVPR 2024)', color='#DD8452', edgecolor='black', linewidth=0.5)

    # Valores sobre barras
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.1f}',
                ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.1f}',
                ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Metrica BOP')
    ax.set_ylabel('Recall (%)')
    ax.set_title(f'{ds_label}')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names)
    ax.set_ylim(0, 100)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Comparativa BOP Metrics: FoundationPose vs GDR-Net++', fontsize=14, fontweight='bold')
plt.tight_layout()

fig_path = f"{FIGURES_DIR}/fp_vs_gdrnet_bop_metrics.png"
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
fig.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f"Figura guardada: {fig_path}")
plt.show()

In [ ]:
from datetime import datetime

RESULTS_DIR = "/content/drive/MyDrive/TFM/experiments/foundationpose_eval"
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

comparison = {
    'timestamp': timestamp,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'config': {
        'max_scenes': MAX_SCENES,
        'max_images_per_scene': MAX_IMAGES_PER_SCENE,
        'register_iterations': REGISTER_ITERATIONS,
    },
    'baselines': {
        'gdrnet': gdrnet_baselines,
        'foundationpose_paper': fp_paper_baselines,
    },
    'our_results': {
        'ycbv': {
            'metrics': metrics_fp_ycbv,
            'n_predictions': len(results_ycbv),
            'timing_total_s': timing_total,
        },
        'tless': {
            'metrics': metrics_fp_tless if 'metrics_fp_tless' in dir() else {},
            'n_predictions': len(results_tless),
            'timing_total_s': timing_tless if 'timing_tless' in dir() else 0,
        },
    },
}

result_file = f"{RESULTS_DIR}/comparison_{timestamp}.json"
with open(result_file, 'w') as f:
    json.dump(comparison, f, indent=2)
print(f"[OK] Resultados guardados: {result_file}")

# Guardar tambien predicciones completas
if results_ycbv:
    preds_file = f"{RESULTS_DIR}/predictions_ycbv_{timestamp}.json"
    with open(preds_file, 'w') as f:
        json.dump({'predictions': results_ycbv, 'metrics': metrics_fp_ycbv}, f, indent=2)
    print(f"[OK] Predicciones YCB-V: {preds_file}")

if results_tless:
    preds_file = f"{RESULTS_DIR}/predictions_tless_{timestamp}.json"
    with open(preds_file, 'w') as f:
        json.dump({'predictions': results_tless, 'metrics': metrics_fp_tless}, f, indent=2)
    print(f"[OK] Predicciones T-LESS: {preds_file}")

print(f"\nPipeline completo. Resultados en Google Drive.")